# How to access Sup3rCC

Sup3rCC is a gridded, downscaled, high-resolution climate dataset created by the National Laboratory of the Rockies. It is intended for use in energy studies. It contains wind, solar, temperature, humidity, and precipitation variables at 4 km spatial resolution and hourly time resolution (except for precipitation, which is daily). Spatially, it covers a large expanse of North America including all of CONUS. The SSP 2-4.5 scenario is used for future projections. To learn more about the downscaling methods employed, see [Buster et al. (2025)](https://doi.org/10.1016/j.dib.2025.111774).

**Is Sup3rCC right for my use case?** This dataset is targeted at users who want to study renewable energy resources (solar, wind, and water) at high spatial resolution for near- to mid-term planning. However, these variables may also be relevant to use cases beyond renewable energy planning. There are a few things to keep in mind when deciding whether to use Sup3rCC: 
  - Only the SSP 2-4.5 scenario is available for projections, so it would not be appropriate for climate vulnerability assessments that require SSP 3-7.0. 
  - Global warming levels are not available for use with this data through climakitae
  - Late-century data is not available for most simulations.
  - Choice of simulations was based on CONUS performance ([Vimmerstedt et al. 2025](https://www.osti.gov/biblio/2498329)), not regional California or WECC performance.

**Intended Application**: As a user, I want to learn how to <span style="color:red">**use climakitae to access Sup3rCC simulations**</span> available on Cal-Adapt.

**Runtime**: This notebook runs in less than one minute. Modifications to queries may increase/decrease the overall runtime.

**References**:  
National Laboratory of the Rockies: [Sup3r: Super-Resolution for Renewable Energy Resource Data](https://www.nlr.gov/analysis/sup3r) (Accessed October 2026)  
Buster, G., Benton, B., Rastogi, D., Kao, S., Castelao, G., and Eisenman., J., 2025: Second-generation downscaled earth system model data using generative machine learning. [doi.org/10.1016/j.dib.2025.111774](https://doi.org/10.1016/j.dib.2025.111774)  
L. Vimmerstedt, B. Benton, G. Buster, S. Podgorny, 2025: Selection of Global Climate Model Data for Downscaling With Generative Machine Learning and Use in the Power Planning for Alignment of Climate and Energy Systems Project. [doi:10.2172/2498329](https://doi.org/10.2172/2498329)


In [ ]:
import climakitae as ck
from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt

In [ ]:
cd = ck.ClimateData(verbosity=-1)

The table below shows the ClimateData options for Sup3rCC.

| Keyword  | Options  | Required?  | Notes  |
|---|---|---|---|
| catalog | "sup3rcc"  | yes  |   |  
| activity_id | "Sup3rCC" | no  |  |  
| source_id | "EC-Earth3-CC", "EC-Earth3-Veg", "GFDL-CM4", "MPI-ESM1-2-HR", "TaiESM1"  | no  | Can leave unset to request all sims  |   
| experiment_id | "historical","ssp245"  | no  | "historical": only years 2000-2014 available. "ssp245": 2015-2059 for all sims, 2015-2099 for TaiESM1  |   
| table_id | "1hr","day"  | yes  | Use "day" for prec, use "1hr" for all other variable_id values  |   
| grid_label | "conus4km"  | yes  |   |   
| varible_id | "psfc", "rh", "swddif", "swddni", "swdnb", "t2", "wd10", "wd100", "wd200", "ws10", "ws100", "ws200", "prec"  | yes  |   |   
| processes | climakitae processors  | no  | Not valid: `warming_level`, `bias_adjust_model_to_station`, `filter_unadjusted_models` | 


## 1. Timeseries
In the example below, hourly air temperature data will be fetched for a single simulation (EC-Earth3-CC). Then `climakitae` processors are applied so the data is clipped to the coordinate 37N, 120W, sliced to years 2001-2003, and converted to use units of degrees Fahrenheit. While a point in California has been selected for this demonstration, any lat/lon coordinates within the Sup3rCC coverage area could be retrieved. **It is recommended to always apply the `clip` processor when possible to reduce the size of the retrieved data.** See the [basic_data_access](https://github.com/cal-adapt/cae-notebooks/blob/main/basic_data_access.ipynb) notebook to learn more about clipping.

In [ ]:
ds = (
    cd.catalog("sup3rcc")
    .source_id("EC-Earth3-CC")  # include source_id to only fetch 1 simulation
    .experiment_id("historical")
    .table_id("1hr")
    .grid_label("conus4km")
    .variable_id("t2")
    .processes(
        {
            "clip": (
                37.0,
                -120.0,
            ),  # Clipping is recommended in all cases. Any clipping option may be used.
            "time_slice": (
                "2001-01-01 00",
                "2003-12-31 23",
            ),  # See valid years in table
            "convert_units": "degF",
        }
    )
).get()

This returns a 3-year timeseries of hourly temperature, which is visualized in the cell below.

**Memory management**: In the cell below, ds.load() is called to read the data into memory before plotting. This is acceptable in this case because the retrieved data is only 0.4 MB in size due to being clipped in space and time. Users should not attempt to load Sup3rCC data for all of CONUS at once. The entire dataset is extremely large in size and will exceed memory limits on the AE Hub.

In [ ]:
# Read data into memory for quicker plotting
with ProgressBar():
    ds.load()

f = plt.figure(figsize=(11, 4))
ds.t2.plot(color="tab:red", linewidth=0.5)

## 2. Mapping

This next example shows how to load county level precipitation data and make a map. Sup3rCC uses a lat/lon coordinate system in EPSG:4326. 

In [ ]:
cd.reset()

ds = (
    cd.catalog("sup3rcc")
    .source_id(
        "EC-Earth3-CC"
    )  # Use 1 sim; attempting to load all sims for a large county will crash the Hub instance
    .experiment_id("ssp245")
    .table_id("day")
    .grid_label("conus4km")
    .variable_id("prec")
    .processes(
        {
            "clip": "San Luis Obispo County",
            "time_slice": ("2020-01-01 00", "2029-12-31 23"),
            "convert_units": "inches",
        }
    )
).get()

This daily data is used to calculate the annual mean precipitation for the ten year time period. The result is displayed below.

In [ ]:
ds.prec.groupby("time.year").sum().mean("year").plot(cmap="BuPu")
plt.title("Mean annual precipitation\nfor San Luis Obispo County (2020-2029)")
plt.xticks([-121, -120.5, -120, -119.5]);  # Tidier lon axis